# ChibiCreate — Notebook 2: modelo como REFINER do FLUX

```
PERSONAGEM ORIGINAL
      ↓
FLUX.2 klein 4B          (já executado — NÃO roda aqui)
      ↓
run_003/output.png
      ↓
MODELO SELECIONADO
      ↓
OUTPUT FINAL
```

Descobre qual modelo funciona melhor como **refiner** do resultado do FLUX.
O `run_003` preserva bem o design mas está pouco chibi; o segundo estágio
existe para puxar as proporções **sem** reinterpretar o design.

**O FLUX não é executado aqui.** Você envia o ZIP com as runs já produzidas e
o `run_003/output.png` entra exatamente como está — sem redimensionar,
recomprimir ou editar.

## Modelos no dropdown

| Modelo | Refs de design | Licença | Comercial |
|---|---|---|---|
| LongCat-Image-Edit | **0** — só a imagem do FLUX | Apache-2.0 | ✅ verified |
| Z-Image Turbo | **0** — img2img da saída FLUX | Apache-2.0 | ✅ verified |
| Qwen-Image-Edit-2511 Q3_K_M | 2 (`full_body`+`outfit`) | Apache-2.0 base | ⚠️ pending review |
| Qwen-Image-Edit-2511 Q4_0 | 2 (`full_body`+`outfit`) | Apache-2.0 base | ⚠️ pending review |
| Pony Diffusion V6 XL | **0** — img2img | FAIPL-1.0-SD mod. | ⛔ **research_only** |

⛔ **Pony: "Research only — not approved for commercial production".**

⚠️ Só o Qwen aceita `full_body` + `outfit` como referências de design. Nos
demais **as referências são descartadas** — isso aparece em destaque e vai
para o recipe. Não é detalhe: muda a leitura do resultado.

**Nada é baixado até você escolher e confirmar.**


In [ ]:
#@title 1 · Setup — clonar repo e carregar o registry { display-mode: "form" }
#@markdown Clona o ChibiCreate, instala dependencias e valida o registry.
#@markdown Rode esta celula primeiro em qualquer runtime novo.
forcar_reclone = False #@param {type:'boolean'}

import os, subprocess, sys, importlib, shutil

REPO_DIR = '/content/ChibiCreate'
SCRIPTS_DIR = REPO_DIR + '/scripts'

if forcar_reclone and os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch',
                    'arena/01a07ece-chibicreate',
                    'https://github.com/BloomRX/ChibiCreate.git', REPO_DIR],
                   check=True)

os.chdir(REPO_DIR)
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyyaml', 'pillow', 'huggingface_hub'], check=True)

# O import falha AQUI, com diagnostico, em vez de virar um NameError
# tres celulas adiante.
try:
    from chibi import model_registry as mr
    importlib.reload(mr)
except Exception as exc:
    raise RuntimeError(
        'FALHA AO IMPORTAR scripts/chibi/model_registry.py\n'
        f'  erro    : {type(exc).__name__}: {exc}\n'
        f'  cwd     : {os.getcwd()}\n'
        f'  arquivo : {os.path.isfile(SCRIPTS_DIR + "/chibi/model_registry.py")}\n'
        'Acao: marque forcar_reclone e rode de novo.') from exc

FALTANDO = [n for n in ('preflight', 'load_registry', 'dropdown_options',
                        'key_for_label', 'get_model', 'prompt_for',
                        'plan_references', 'run_dir_for', 'describe',
                        'needs_confirmation', 'comparison_table')
            if not hasattr(mr, n)]
if FALTANDO:
    raise RuntimeError('registry incompleto, faltam: ' + ', '.join(FALTANDO)
                       + '. Marque forcar_reclone e rode de novo.')

REG = mr.load_registry()
SETUP_OK = True

print('SETUP OK')
print('  repo     :', REPO_DIR)
print('  registry :', len(REG['models']), 'modelos')
print()
print('Proxima celula: escolher o modelo no dropdown.')


In [ ]:
#@title 2 · Escolher modelo { display-mode: "form" }
#@markdown Selecione o modelo e rode. Nada e baixado nesta celula.
modelo = 'Qwen-Image-Edit-2511 Q3_K_M' #@param ['LongCat-Image-Edit', 'Z-Image Turbo', 'Qwen-Image-Edit-2511 Q3_K_M', 'Qwen-Image-Edit-2511 Q4_0', 'Pony Diffusion V6 XL (research only)']
seed = 42 #@param {type:'integer'}

if not globals().get('SETUP_OK'):
    raise RuntimeError('Rode a celula 1 (Setup) antes desta.')

MODEL_LABEL = modelo
MODEL_KEY = mr.key_for_label(MODEL_LABEL, REG)
CFG = mr.get_model(MODEL_KEY, REG)
PARAMS = CFG['parameters']

PROMPT_INFO = mr.prompt_for(MODEL_KEY, REG)
PROMPT = PROMPT_INFO['prompt']
NEGATIVE_PROMPT = PROMPT_INFO['negative_prompt']
SEED = seed
SELECTION_OK = True

print(mr.describe(MODEL_KEY, REG))
print()
print('PROMPT:', len(PROMPT), 'caracteres')
if PROMPT_INFO['override_applied']:
    print('  override do modelo:', PROMPT_INFO['override_reason'])
print('NEGATIVE PROMPT:', repr(NEGATIVE_PROMPT), '(sem negativas automaticas)')
print('SEED:', SEED)


In [ ]:
#@title 3 · Preflight — GPU, VRAM e disco { display-mode: "form" }
#@markdown Mede o ambiente real e decide READY / BLOCKED **antes** de baixar
#@markdown qualquer coisa. Em bloqueio, para e nao baixa nada.

_faltando = [n for n in ('mr', 'REG', 'MODEL_KEY', 'CFG') if n not in globals()]
if _faltando:
    raise RuntimeError(
        'Faltam variaveis: ' + ', '.join(_faltando) + '\n'
        '  mr / REG        -> celula 1 (Setup)\n'
        '  MODEL_KEY / CFG -> celula 2 (Escolher modelo)')

import shutil, subprocess, sys


def _gpu():
    """Le a GPU real. Nao assume T4/L4/A100."""
    try:
        out = subprocess.run(
            ['nvidia-smi',
             '--query-gpu=name,memory.total,memory.free,driver_version',
             '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None
    if not out:
        return None
    nome, total, livre, drv = [x.strip()
                               for x in out.split('\n')[0].split(',')[:4]]
    return {'name': nome, 'vram_total_gb': float(total) / 1024,
            'vram_free_gb': float(livre) / 1024, 'driver': drv}


GPU = _gpu()
_du = shutil.disk_usage('/content')
DISK_FREE_GB = _du.free / 1024 ** 3
DISK_TOTAL_GB = _du.total / 1024 ** 3

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1024 ** 3
except Exception:
    RAM_GB = None

try:
    import torch
    TORCH_V, CUDA_V = torch.__version__, torch.version.cuda
except Exception:
    TORCH_V = CUDA_V = None

print('AMBIENTE REAL (medido, nao assumido)')
print('  GPU        :', GPU['name'] if GPU else 'NENHUMA')
print('  VRAM total :', f"{GPU['vram_total_gb']:.1f} GB" if GPU else '-')
print('  VRAM livre :', f"{GPU['vram_free_gb']:.1f} GB" if GPU else '-')
print('  RAM        :', f'{RAM_GB:.1f} GB' if RAM_GB else 'desconhecida')
print('  Disk total :', f'{DISK_TOTAL_GB:.1f} GB')
print('  Disk free  :', f'{DISK_FREE_GB:.1f} GB')
print('  CUDA       :', CUDA_V)
print('  Python     :', sys.version.split()[0])
print('  PyTorch    :', TORCH_V)
print()

PF = mr.preflight(MODEL_KEY, available_disk_gb=DISK_FREE_GB,
                  available_vram_gb=GPU['vram_free_gb'] if GPU else None,
                  registry=REG)
print(PF.report())

PREFLIGHT_OK = PF.ready
if not PF.ready:
    # Parar e o comportamento correto: o requisito do modelo NAO e
    # reduzido para caber no runtime.
    raise SystemExit(
        'PARE: ' + PF.status + '\n'
        'Nenhum download foi iniciado.\n'
        'Opcoes: escolher outro modelo (celula 2), trocar de runtime '
        '(Runtime > Change runtime type) ou rodar o cleanup.')


In [ ]:
#@title 4 · Autorizar download { display-mode: "form" }
#@markdown Marque a caixa para autorizar. Downloads grandes, quantizacoes de
#@markdown terceiros e modelos research-only exigem aceite explicito.
autorizo_o_download = False #@param {type:'boolean'}

if not globals().get('PREFLIGHT_OK'):
    raise RuntimeError('Rode o preflight (celula 3) antes desta.')

if mr.needs_confirmation(MODEL_KEY, REG):
    print('!' * 64)
    print('CONFIRMACAO NECESSARIA')
    print(f"  download estimado : {CFG['download_gb']} GB")
    if CFG.get('requires_custom_node'):
        print(f"  custom node       : {CFG['requires_custom_node']}")
        print('  ' + ' '.join(CFG['custom_node_note'].split()))
    if CFG.get('commercial_status') == 'research_only':
        print('  ' + CFG['commercial_banner'])
    print('!' * 64)
    print()

CONFIRMADO = bool(autorizo_o_download)
if not CONFIRMADO:
    raise SystemExit(
        'Download NAO autorizado. Marque "autorizo_o_download" e rode de '
        'novo. Nada foi baixado.')

print('Autorizado baixar:', MODEL_LABEL)


In [ ]:
#@title 5 · Upload do ZIP com as runs do FLUX { display-mode: "form" }
from google.colab import files
import zipfile, pathlib, shutil, hashlib
from PIL import Image
from IPython.display import display

dest = pathlib.Path('/content/flux_input')
if dest.exists():
    shutil.rmtree(dest)
dest.mkdir(parents=True)

print('Selecione o ZIP com os resultados do FLUX...')
up = files.upload()
zip_name = list(up)[0]
with zipfile.ZipFile(zip_name) as z:
    z.extractall(dest)

print()
print('saidas encontradas no ZIP:')
for p in sorted(dest.rglob('run_*/output.png')):
    print('  ', p.relative_to(dest))

hits = sorted(dest.rglob('run_003/output.png'))
if not hits:
    raise SystemExit(
        'PARE: run_003/output.png nao encontrado. Ele e o ponto de partida '
        'do segundo estagio e nao pode ser substituido por outro run.')
if len(hits) > 1:
    raise SystemExit(f'PARE: {len(hits)} candidatos a run_003. Ambiguo.')

FLUX_RUN003 = hits[0].resolve()
print()
print('entrada do estagio 2:', FLUX_RUN003)


In [ ]:
#@title 6 · Verificar imagem do FLUX (hashes) { display-mode: "form" }
# Imports proprios: esta celula nao pode depender da de upload ter
# rodado no mesmo kernel.
import hashlib, pathlib
from PIL import Image
from IPython.display import display

if 'FLUX_RUN003' not in globals():
    raise RuntimeError('Rode a celula de upload do ZIP (4) antes desta.')

data = FLUX_RUN003.read_bytes()
with Image.open(FLUX_RUN003) as im:
    wh, modo = im.size, im.mode
    px = hashlib.sha256(im.convert('RGBA').tobytes()).hexdigest()

STAGE1 = {
    'file': str(FLUX_RUN003),
    'source_run': 'run_003',
    'origin': f'upload do usuario: {zip_name} -> run_003/output.png',
    'artifact_sha256': hashlib.sha256(data).hexdigest(),
    'pixel_sha256': px,
    'width': wh[0], 'height': wh[1], 'mode': modo, 'bytes': len(data),
    'modified_before_stage2': False,
    'flux_reexecuted': False,
}
for k, v in STAGE1.items():
    print(f'  {k:24} {v}')

display(Image.open(FLUX_RUN003))


In [ ]:
#@title 7 · Referencias suportadas pelo modelo { display-mode: "form" }
import hashlib, pathlib
from PIL import Image

if 'FLUX_RUN003' not in globals():
    raise RuntimeError('Rode a celula de upload do ZIP (4) antes desta.')

REF_DIR = pathlib.Path('characters/waifu_001/reference')

def ficha(p):
    d = p.read_bytes()
    with Image.open(p) as im:
        return {'file': p.name, 'path': str(p),
                'artifact_sha256': hashlib.sha256(d).hexdigest(),
                'pixel_sha256': hashlib.sha256(
                    im.convert('RGBA').tobytes()).hexdigest(),
                'width': im.size[0], 'height': im.size[1], 'bytes': len(d)}

DESEJADAS, REF_FILES = [], {}
for nome in ('full_body.png', 'outfit.png'):
    p = REF_DIR / nome
    if p.is_file():
        DESEJADAS.append(nome)
        REF_FILES[nome] = ficha(p)

PRIMARY = str(FLUX_RUN003)
PRIMARY_ROLE = 'stage1_output'

PLANO = mr.plan_references(MODEL_KEY, PRIMARY, PRIMARY_ROLE, DESEJADAS,
                           registry=REG)
print(PLANO.report())

if PLANO.has_dropped:
    print()
    print('=' * 64)
    print('LIMITACAO IMPORTANTE PARA A LEITURA DO RESULTADO')
    print('Este modelo recebe SOMENTE a saida do FLUX, sem as referencias')
    print('de design. Se o design se perder, pode ser falta de referencia,')
    print('nao necessariamente fraqueza do modelo.')
    print('=' * 64)


In [ ]:
#@title 8 · Baixar o modelo selecionado { display-mode: "form" }
# Baixa APENAS o modelo selecionado. Repos e arquivos vem do registry.
from huggingface_hub import hf_hub_download, snapshot_download
import hashlib, pathlib, time

if not PF.ready or not CONFIRMADO:
    raise SystemExit('preflight nao aprovado ou download nao confirmado.')

DEST = pathlib.Path('/content/models') / MODEL_KEY
DEST.mkdir(parents=True, exist_ok=True)
t0 = time.time()

baixados = []
if CFG.get('file'):
    p = hf_hub_download(repo_id=CFG['repo'], filename=CFG['file'],
                        revision=CFG.get('revision'), local_dir=str(DEST))
    baixados.append(pathlib.Path(p))
else:
    d = snapshot_download(repo_id=CFG['repo'], revision=CFG.get('revision'),
                          local_dir=str(DEST))
    baixados = [q for q in pathlib.Path(d).rglob('*')
                if q.is_file() and q.suffix in {'.safetensors', '.gguf'}]

MODEL_RECORD = {
    'model_key': MODEL_KEY,
    'label': MODEL_LABEL,
    'repo': CFG['repo'],
    'revision_requested': CFG.get('revision'),
    'revision_verified': CFG.get('revision_verified', False),
    'license': CFG.get('license'),
    'license_verified': CFG.get('license_verified'),
    'commercial_status': CFG.get('commercial_status'),
    'third_party_quantization': CFG.get('third_party_quantization', False),
    'quantization_license': CFG.get('quantization_license'),
    'quantization_author': CFG.get('quantization_author'),
    'download_seconds': round(time.time() - t0, 1),
    'files': [],
}
for f in baixados:
    h = hashlib.sha256()
    with open(f, 'rb') as fh:
        for bloco in iter(lambda: fh.read(1 << 22), b''):
            h.update(bloco)
    MODEL_RECORD['files'].append(
        {'name': f.name, 'bytes': f.stat().st_size, 'sha256': h.hexdigest()})
    print(f'  {f.name}  {f.stat().st_size / 1e9:.2f} GB  {h.hexdigest()[:16]}...')

print()
print('baixado em', MODEL_RECORD['download_seconds'], 's')
print('AVISO: sha256 acima e do arquivo COMO BAIXADO. O registry nao tinha')
print('hash previo para conferir, entao isto e registro, nao verificacao.')


In [ ]:
#@title 9 · Executar (uma vez) { display-mode: "form" }
import time, json, pathlib, hashlib

RUN_DIR = mr.run_dir_for(mr.STAGE_FLUX_REFINER, MODEL_KEY,
                         pathlib.Path('/content/ChibiCreate'), REG)
print('run em', RUN_DIR)

RECIPE = {
    'stage': mr.STAGE_FLUX_REFINER,
    'pipeline': 'original -> flux run_003 -> model',
    'stage1': STAGE1,
    'flux_reexecuted': False,
    'model_key': MODEL_KEY,
    'label': MODEL_LABEL,
    'pipeline_type': CFG['pipeline_type'],
    'input_mode': CFG['input_mode'],
    'model': MODEL_RECORD,
    'commercial_status': CFG['commercial_status'],
    'excluded_from_commercial_ranking': CFG.get(
        'excluded_from_commercial_ranking', False),
    'prompt': PROMPT,
    'negative_prompt': NEGATIVE_PROMPT,
    'prompt_override_applied': PROMPT_INFO['override_applied'],
    'seed': SEED,
    'steps': PARAMS['steps'],
    'cfg': PARAMS['cfg'],
    'sampler': PARAMS['sampler'],
    'scheduler': PARAMS['scheduler'],
    'denoise': PARAMS.get('denoise'),
    'denoise_status': PARAMS.get('denoise_status'),
    'resolution': PARAMS['resolution'],
    'batch': PARAMS['batch'],
    'references': PLANO.to_dict(),
    'reference_files': REF_FILES,
    'environment': {
        'gpu': GPU['name'] if GPU else None,
        'vram_total_gb': round(GPU['vram_total_gb'], 2) if GPU else None,
        'cuda': CUDA_V, 'torch': TORCH_V,
        'ram_gb': round(RAM_GB, 1) if RAM_GB else None,
        'infrastructure': 'google_colab',
        'infrastructure_status': 'EXPERIMENTAL_TEMPORARY',
    },
    'limitations': [
        'uma execucao: nao afirma determinismo',
        'seed nao atravessa estagios: nao reproduz o ruido do FLUX',
        'requisitos de VRAM/disco vem do model card, nao de medicao nossa',
    ],
    'approval_status': 'experimental',
}
if PLANO.has_dropped:
    RECIPE['limitations'].append(
        'SEM referencias de design (' + ', '.join(PLANO.dropped)
        + '): o modelo recebeu apenas a saida do FLUX.')
if CFG.get('capability_warning'):
    RECIPE['limitations'].append(' '.join(CFG['capability_warning'].split()))

t0 = time.time()
# ------------------------------------------------------------------
# [HUMAN REVIEW REQUIRED] / [TEST REQUIRED]
# Mesma situacao do Notebook 1: a chamada de inferencia varia por pipeline
# e nenhuma foi executada contra GPU real neste ambiente.
# ------------------------------------------------------------------
raise NotImplementedError(
    'Adapter de inferencia ainda nao implementado para ' + MODEL_KEY + '.\n'
    'Preflight, upload do run_003, hashes, plano de referencias e recipe '
    'acima ja funcionam.')


In [ ]:
#@title 10 · Fechar o recipe { display-mode: "form" }
# Fecha o recipe depois da inferencia.
import hashlib, json

OUT = RUN_DIR / 'output.png'
RECIPE['execution_time'] = round(time.time() - t0, 1)
if OUT.is_file():
    RECIPE['artifact_sha256'] = hashlib.sha256(OUT.read_bytes()).hexdigest()
    with Image.open(OUT) as im:
        RECIPE['output_pixel_sha256'] = hashlib.sha256(
            im.convert('RGBA').tobytes()).hexdigest()
    RECIPE['pixel_hash_note'] = (
        'output_pixel_sha256 e o hash dos PIXELS; artifact_sha256 e o hash '
        'do ARQUIVO.')

(RUN_DIR / 'recipe.json').write_text(
    json.dumps(RECIPE, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(RECIPE, indent=2, ensure_ascii=False)[:1600])


In [ ]:
#@title 11 · Tabela de comparacao { display-mode: "form" }
import pathlib, json

print('=' * 64); print('ORIGINAL')
display(Image.open('characters/waifu_001/reference/full_body.png').resize((256, 256)))
print('=' * 64); print('ESTAGIO 1 — FLUX run_003 (entrada)')
display(Image.open(FLUX_RUN003).resize((256, 256)))

base = pathlib.Path('experiments/model_eval/flux_to_model')
linhas = []
for rec_path in sorted(base.rglob('recipe.json')):
    rec = json.loads(rec_path.read_text())
    linhas.append({
        'model': rec.get('label', rec.get('model_key')),
        'time': rec.get('execution_time'),
        'vram': rec.get('environment', {}).get('vram_total_gb'),
        'status': rec.get('commercial_status'),
    })
    out = rec_path.parent / 'output.png'
    if out.is_file():
        print('=' * 64)
        print(rec.get('label'), '|', rec.get('references', {}).get(
            'references_used') or 'sem referencias de design')
        display(Image.open(out).resize((256, 256)))

print()
print(mr.comparison_table(linhas))


## Avaliação — os três eixos

Preencha à mão. **Não há OVERALL** e não é média aritmética.

| Eixo | O que olhar |
|---|---|
| **DESIGN_PRESERVATION** *(eixo principal)* | roupa, capa, ornamentos, acessórios, chifres |
| **IDENTITY** | rosto, cabelo, cores, é a mesma personagem? |
| **STYLE** | silhueta, proporções chibi, leitura em tamanho pequeno |

Percorra o checklist item a item:

roupa · capa · ornamentos · acessórios · chifres · cabelo · rosto ·
silhueta · proporções chibi · preservação das cores ·
**detalhes inventados ou removidos**

*Simplificar é remover detalhe. Redesenhar é trocar o design.* Um chibi mais
limpo não é perda de design; uma capa que virou outra capa é.

**A decisão artística é humana.** O agente não escolhe vencedor, não aprova
Chibi Master e não julga beleza. Um resultado do Pony **nunca** altera o
candidato comercial.


In [ ]:
#@title 12 · Exportar resultados (antes do reset) { display-mode: "form" }
# Exporta os resultados ANTES de qualquer reset.
import shutil, pathlib, json

d = pathlib.Path('experiments/model_eval/flux_to_model')
if not d.exists():
    raise SystemExit('nada a exportar ainda.')

shutil.make_archive('/content/flux_to_model_results', 'zip', d)
z = pathlib.Path('/content/flux_to_model_results.zip')
print('zip:', round(z.stat().st_size / 1e6, 1), 'MB')

from google.colab import files
files.download(str(z))
print()
print('Baixe o arquivo antes de resetar o runtime.')


In [ ]:
#@title 13 · Cleanup opcional (nao apaga experiments/) { display-mode: "form" }
# CLEANUP OPCIONAL — libera espaco antes de trocar de modelo.
# NAO apaga experiments/: os resultados anteriores ficam intactos.
import shutil, pathlib

ALVOS = [
    pathlib.Path('/content/models'),
    pathlib.Path.home() / '.cache' / 'huggingface',
]

for alvo in ALVOS:
    if alvo.exists():
        tam = sum(f.stat().st_size for f in alvo.rglob('*') if f.is_file())
        shutil.rmtree(alvo, ignore_errors=True)
        print(f'  removido {alvo}  ({tam / 1e9:.1f} GB)')
    else:
        print(f'  ausente  {alvo}')

print()
print('experiments/ NAO foi tocado — resultados preservados.')
print('Baixe o ZIP de resultados ANTES de resetar o runtime.')
print('disco livre agora:', round(shutil.disk_usage('/content').free / 1e9, 1), 'GB')


---

**PARE.** Analise, baixe o ZIP, resete o runtime e escolha outro modelo.

A matriz completa responde: qual modelo é melhor **sozinho** (Notebook 1) e
qual é melhor como **refiner do FLUX** (Notebook 2). Só comparando os dois é
possível dizer se `FLUX + modelo` supera o FLUX puro — ou se não melhora nada.

Não é Flow 02. Não produz `master.png`. Nenhum artefato é aprovado.
